# 02 — Ingestion position des aéronefs (OpenSky Network)

**Objectif** : récupérer les positions live des aéronefs au-dessus de la France via l'API REST *OpenSky Network* (`/states/all`), marquer les moyens de la Sécurité Civile, puis écrire dans une table Delta *bronze*.

> **Authentification** : OAuth2 *client credentials* si des identifiants sont fournis, sinon **fallback anonyme** (fortement rate-limité).

> **Callsigns Sécurité Civile** : `PELICAN` = Canadair CL-415, `MILAN` = Dash-8, `DRAGON` = hélicoptères. **Hors saison** feux, il peut n'y avoir aucun bombardier en vol → le notebook 03 fournit un fallback simulé.

In [ ]:
# Cell: Paramètres
OPENSKY_CLIENT_ID     = ""   # OAuth2 client credentials (laisser vide pour accès anonyme, rate-limité)
OPENSKY_CLIENT_SECRET = ""
BBOX = dict(lamin=41.0, lomin=-5.0, lamax=51.5, lomax=10.0)   # France métropolitaine
FIRE_CALLSIGNS = ("PELICAN", "MILAN", "DRAGON")               # Canadair, Dash-8, hélicos Sécurité Civile
LAKEHOUSE_TABLE = "bronze_aircraft_positions"

In [ ]:
# Cell: Authentification (OAuth2 si identifiants fournis)
import requests
token = None
if OPENSKY_CLIENT_ID and OPENSKY_CLIENT_SECRET:
    tok = requests.post(
        "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token",
        data={"grant_type": "client_credentials",
              "client_id": OPENSKY_CLIENT_ID, "client_secret": OPENSKY_CLIENT_SECRET}, timeout=30)
    tok.raise_for_status()
    token = tok.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"} if token else {}

In [ ]:
# Cell: Récupération des états (positions live)
import pandas as pd
from datetime import datetime, timezone
r = requests.get("https://opensky-network.org/api/states/all", params=BBOX, headers=headers, timeout=60)
r.raise_for_status()
cols = ["icao24","callsign","origin_country","time_position","last_contact","longitude","latitude",
        "baro_altitude","on_ground","velocity","true_track","vertical_rate","sensors","geo_altitude",
        "squawk","spi","position_source"]
states = r.json().get("states") or []
df = pd.DataFrame([s[:17] for s in states], columns=cols)
df["callsign"] = df["callsign"].astype(str).str.strip()
print(f"{len(df)} aéronefs dans la zone.")

In [ ]:
# Cell: Marquage des moyens Sécurité Civile + écriture Delta
df["is_firefighting"] = df["callsign"].str.upper().str.startswith(FIRE_CALLSIGNS)
df["aircraft_role"] = df["callsign"].str.upper().str.extract(r"^(PELICAN|MILAN|DRAGON)")
df["ingest_ts"] = datetime.now(timezone.utc).isoformat()
sdf = spark.createDataFrame(df)
(sdf.write.format("delta").mode("append").saveAsTable(LAKEHOUSE_TABLE))
print(f"Écrit dans {LAKEHOUSE_TABLE}. Note: hors saison, peu/pas de bombardiers en vol -> notebook 03 fournit un fallback simulé.")